# AI4I 2020 PMD Ingest Pipeline
This notebook is desribed how ingest-AI4I-2020-PMD.py functions. It is used to pull the data provided (DOI 10.24432/C5HS5C) onto the local system through the use of UCI's mlrepo library, and provide simple verification of the data received

1. Imports and fetch

In [103]:
# imports
import pandas as pd                 # access to pandas feature set
import pyarrow as pa                # allows for saving to parquet (wasn't sure if we wanted to use that or regular csv)
import pyarrow.parquet as pq        # allows for accessing parquet files to check for consistency
from pathlib import Path            # cross-os compatability
from ucimlrepo import fetch_ucirepo # uc irvine library to fetch and format dataset


In [104]:
# establish save directories
RAW_DATA_DIR = Path("../data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True) # create folder if needed

PROC_DATA_DIR = Path("../data/processed")
PROC_DATA_DIR.mkdir(parents=True, exist_ok=True) # create folder if needed

In [ ]:
# AI4I 2020 PMD dataset: 601
DATASET_ID = 601

# fetch dataset
ai4i_2020_pmd = fetch_ucirepo(id=DATASET_ID) 
  
# data (as pandas dataframes)
data = pd.DataFrame(ai4i_2020_pmd.data.original) 
  
# metadata 
print(ai4i_2020_pmd.metadata) 
  
# variable information 
print(ai4i_2020_pmd.variables) 

{'uci_id': 601, 'name': 'AI4I 2020 Predictive Maintenance Dataset', 'repository_url': 'https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/601/data.csv', 'abstract': 'The AI4I 2020 Predictive Maintenance Dataset is a synthetic dataset that reflects real predictive maintenance data encountered in industry.', 'area': 'Computer Science', 'tasks': ['Classification', 'Regression', 'Causal-Discovery'], 'characteristics': ['Multivariate', 'Time-Series'], 'num_instances': 10000, 'num_features': 6, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], 'index_col': ['UID', 'Product ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Wed Feb 14 2024', 'dataset_doi': '10.24432/C5HS5C', 'creators': [], 'intro_paper': {'ID': 386, 'type': 'NATIVE', 'title': 'Explainable Artificial Intelligen

2. Verify data fetched

In [106]:
print(data.info() , f'\n')
print("Total missing values:", data.isnull().sum().sum(),)
data.tail(26)

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   UID                  10000 non-null  int64  
 1   Product ID           10000 non-null  str    
 2   Type                 10000 non-null  str    
 3   Air temperature      10000 non-null  float64
 4   Process temperature  10000 non-null  float64
 5   Rotational speed     10000 non-null  int64  
 6   Torque               10000 non-null  float64
 7   Tool wear            10000 non-null  int64  
 8   Machine failure      10000 non-null  int64  
 9   TWF                  10000 non-null  int64  
 10  HDF                  10000 non-null  int64  
 11  PWF                  10000 non-null  int64  
 12  OSF                  10000 non-null  int64  
 13  RNF                  10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
memory usage: 1.1 MB
None 

Total missing values: 0


,UID,Product ID,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
9974,9975,L57154,L,298.6,308.2,1361,68.2,172,1,0,0,1,1,0
9975,9976,M24835,M,298.6,308.3,1589,34.1,174,0,0,0,0,0,0
9976,9977,L57156,L,298.6,308.2,1705,26.3,177,0,0,0,0,0,0
9977,9978,L57157,L,298.6,308.3,1422,40.4,179,0,0,0,0,0,0
9978,9979,H39392,H,298.6,308.3,1377,52.1,181,0,0,0,0,0,0
9979,9980,L57159,L,298.5,308.3,1334,49.8,186,0,0,0,0,0,0
9980,9981,L57160,L,298.6,308.4,1559,36.6,188,0,0,0,0,0,0
9981,9982,L57161,L,298.7,308.5,1353,47.0,190,0,0,0,0,0,0
9982,9983,M24842,M,298.7,308.4,1507,45.7,192,0,0,0,0,0,0
9983,9984,L57163,L,298.7,308.4,1694,28.7,195,0,0,0,0,0,0


In [107]:
data["Product ID"].nunique()

10000

All Product IDs are unique. With this information, I can exclude the entire column from the model's dataset

3. Save dataset to /data/raw

The dataset will be saved as both a csv file, and as a schema enforced parquet file.
We will likely use the CSVs, but having a direct representation of the original doesn't hurt

In [108]:
# store datasets to data/raw (in both csv and parquet format)
data.to_csv(RAW_DATA_DIR / "ai4i2020.csv")
print(f"Original dataset saved to: {Path(RAW_DATA_DIR / "ai4i2020.csv")}")

data.to_parquet(RAW_DATA_DIR / "ai4i2020.parquet")
print(f"Original dataset saved to: {Path(RAW_DATA_DIR / "ai4i2020.parquet")}")

Original dataset saved to: ../data/raw/ai4i2020.csv
Original dataset saved to: ../data/raw/ai4i2020.parquet


4. Remove unneeded columns, examine feature engineering opportunities

In [109]:
# dropping UID and Product ID columns, as well as targets (twf, hdf, pwf, osf, and rnf)
model_data = data.drop(columns=["UID","Product ID","TWF","HDF","PWF","OSF","RNF"])

In [110]:
# one-hot encoding type to (H)igh, (M)edium, and (L)ow 
# Note: I'm using drop_first=False. This creates perfect multicolinearity. Linear models will struggle
# CAN BE CHANGED TO ORDINAL ENCODING FOR RANDOM FOREST MODEL
model_data = pd.get_dummies(model_data, columns=["Type"], drop_first=False)

In [111]:
# format column names to snake_case
model_data.columns = model_data.columns.str.lower().str.replace(" ", "_")
model_data.head()

,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,type_h,type_l,type_m
0,298.1,308.6,1551,42.8,0,0,False,False,True
1,298.2,308.7,1408,46.3,3,0,False,True,False
2,298.1,308.5,1498,49.4,5,0,False,True,False
3,298.2,308.6,1433,39.5,7,0,False,True,False
4,298.2,308.7,1408,40.0,9,0,False,True,False


5. Save processed data

In [ ]:
# store datasets to data/processed as csv
print(f"Processed dataset saved to: {Path(PROC_DATA_DIR / "ai4i2020_processed.csv")}")

Processed dataset saved to: ../data/processed/ai4i2020_processed.csv
